# Agents: Laboratory Exercise
## Constructing a Goal-Based Agent using a Large Language Model



### Task 1 - Understanding the Problem

**1. What is the environment?**
The environment is a discrete, deterministic, fully observable, and static 2D grid representing a warehouse. It consists of free navigable spaces (`.`), impassable obstacles (`#`), a starting point (`S`), and a goal destination (`G`).

**2. What is the goal of the agent?**
The goal of the agent is to navigate from the starting position (`S`) to the goal position (`G`) while avoiding all obstacles (`#`) and staying within the bounds of the warehouse.

**3. What actions are available to the agent?**
The agent has four discrete actions available: move **Up**, **Down**, **Left**, and **Right** by exactly one grid square per move.

**4. What information must the agent maintain in order to choose its next action?**
The agent must maintain an internal model of the world which includes:
- The map of the environment (locations of obstacles).
- Its current state (current X, Y coordinates).
- The goal state (target X, Y coordinates).
- Depending on the planning algorithm, it must also maintain a memory of visited states (to avoid infinite loops) and a frontier of states to explore.

**5. Why is this an example of a goal-based agent rather than a simple reflex agent?**
A simple reflex agent only acts based on the current percept (e.g., "if there is a wall in front, turn left"). It cannot look ahead. Because this maze requires finding a path to a specific destination that isn't immediately visible, the agent must plan a sequence of actions by considering future consequences to achieve an explicit objective. This requires reasoning about the goal, making it a goal-based agent.

---
**Think About It**
*Suppose the warehouse becomes twice as large. Would the same search strategy still be appropriate? What additional difficulties might arise?*
While the same fundamental search strategy (like Breadth-First Search) would still mathematically work, it might become computationally inappropriate due to the explosion in the state space. BFS explores in all directions equally. In a map twice as large, the number of nodes to explore grows quadratically, leading to significantly higher memory usage and execution time. To overcome this, it would be necessary to switch to an *informed* search strategy like A* (A-Star), which uses a heuristic (like Manhattan distance) to direct the search preferentially towards the goal.



### Task 2 - Designing the Agent

**Agent Design Components:**
- **Environment**: The 2D character grid of the warehouse.
- **Current State**: The `(row, column)` coordinates representing the vehicle's current location.
- **Goal**: Reaching the `(row, column)` coordinates of the dispatch area `G`.
- **Available Actions**: Directions vector: `(0, 1)`, `(0, -1)`, `(1, 0)`, `(-1, 0)` corresponding to Right, Left, Down, Up.
- **Decision-Making Component**: A pathfinding algorithm (e.g., Breadth-First Search) that simulates actions in its internal model of the environment to build a sequence of moves to the goal before executing them.

**Block Diagram of Goal-Based Agent (Mermaid):**
```mermaid
flowchart TD
    Env[Environment: 2D Grid Maze] -->|Percept: Current Position & Map| Agent
    
    subgraph Goal-Based Agent
        State[Current State]
        Goal[Goal State]
        Model[Internal World Model]
        Plan[Pathfinding Algorithm / Planner]
        
        State --> Plan
        Goal --> Plan
        Model --> Plan
    end
    
    Plan -->|Action Sequence: Up/Down/Left/Right| Env
```



### Task 3 - Prompt Engineering

**Prompt used to generate the code:**
> Write a well-documented Python program implementing a goal-based agent for the warehouse navigation problem. The map is provided below:
> #####################
> #S....#............G#
> #.##....##########..#
> #....##.............#
> #.######.###.#.###..#
> #........#..........#
> #####################
> 
> The program should:
> - represent the warehouse as a two-dimensional grid;
> - determine a collision-free path from S to G;
> - avoid all obstacles;
> - print either the path found (visually mapping it out) or a suitable message if no path exists;
> - explain the search algorithm that has been chosen and why it is appropriate.



In [ ]:
from collections import deque

# The warehouse map represented as a list of strings
warehouse_map = [
    "#####################",
    "#S....#............G#",
    "#.##....##########..#",
    "#....##.............#",
    "#.######.###.#.###..#",
    "#........#..........#",
    "#####################"
]

def find_start_and_goal(grid):
    start, goal = None, None
    for r, row in enumerate(grid):
        for c, char in enumerate(row):
            if char == 'S':
                start = (r, c)
            elif char == 'G':
                goal = (r, c)
    return start, goal

def solve_warehouse(grid):
    start, goal = find_start_and_goal(grid)
    if not start or not goal:
        print("Error: Start or Goal not found in the map.")
        return

    # Dimensions
    rows = len(grid)
    cols = len(grid[0])

    # Allowed moves: Up, Down, Left, Right
    directions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    
    # BFS Queue: stores tuples of (current_position, path_taken_so_far)
    queue = deque([(start, [start])])
    
    # Visited set to avoid cycles
    visited = {start}

    # BFS Loop
    while queue:
        current_pos, path = queue.popleft()
        r, c = current_pos

        # Check if goal is reached
        if current_pos == goal:
            print(f"Goal Reached! Path length: {len(path)-1} moves.")
            return path

        # Explore neighbors
        for dr, dc in directions:
            nr, nc = r + dr, c + dc
            
            # Check bounds and obstacles
            if 0 <= nr < rows and 0 <= nc < cols:
                if grid[nr][nc] != '#' and (nr, nc) not in visited:
                    visited.add((nr, nc))
                    queue.append(((nr, nc), path + [(nr, nc)]))
                    
    print("No collision-free path exists.")
    return None

def visualize_path(grid, path):
    if not path:
        return
    
    # Convert string grid to list of lists for mutability
    grid_list = [list(row) for row in grid]
    
    # Mark the path with '*' (excluding Start and Goal)
    for r, c in path[1:-1]:
        grid_list[r][c] = '*'
        
    print("\nVisualized Path (* denotes the path taken):")
    for row in grid_list:
        print("".join(row))

# Run the agent
print("Algorithm Chosen: Breadth-First Search (BFS)")
print("Reasoning: BFS explores the grid level by level. In an unweighted grid like this warehouse (where every move costs exactly 1), BFS guarantees finding the absolute shortest path from S to G.\n")

path = solve_warehouse(warehouse_map)
visualize_path(warehouse_map, path)



### Evaluating the LLM Output

**1. Did the LLM generate a working program on the first attempt?**
Yes, the provided code effectively models the environment, executes Breadth-First Search, safely navigates around obstacles without IndexErrors, and successfully prints the visual path mapping.

**2. If not, how can you improve your prompt?**
*(Not applicable since it worked, but generally speaking)*: Providing explicit corner-cases in the prompt, such as "make sure to not step out of array bounds" or "ensure the start and goal positions are dynamically searched rather than hardcoded", guarantees higher first-pass success.

**3. What search algorithm did the LLM choose?**
The LLM chose Breadth-First Search (BFS) implemented via a FIFO queue (`collections.deque`).

**4. Why do you think the LLM selected this algorithm?**
BFS is the textbook standard and most appropriate un-informed search algorithm for finding the shortest path on an unweighted 2D grid. Because each step costs exactly 1 unit, BFS perfectly guarantees optimality (shortest path) without the added complexity of implementing heuristic functions required by A* (A-Star). Given the small size of the map, BFS is highly efficient.

